In [48]:
import pandas as pd
import numpy as np
import re
import xgboost as xgb

In [49]:
DATA_FOLDER = "./"

df = pd.read_pickle(DATA_FOLDER + "df_fe_epic_big_best_customers_10c.pickle")
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()


In [50]:
transformations = {
    "tn": [
        r"tn$",
        r"cust_request_qty_per_tn$",
        r"tn_lag_*",
        r"tn_rolling_mean_*",
        r"tn_rolling_max_*",
        r"tn_rolling_min_*",
        r"tn_.*_vendidas$",
        r"tn_agg*",
    ]
    + [r"stock_final$"]
    + [r"cust_request_tn_minus_tn$"]
    + [r"tn_diff_*"],
    "cust_request_qty": [
        r"cust_request_qty$",
        r"cust_request_qty_lag_*",
        r"cust_request_qty_rolling_mean_*",
        r"cust_request_qty_rolling_max_*",
        r"cust_request_qty_rolling_min_*",
        r"cust_request_qty_.*_vendidas$",
        r"cust_request_qty_agg*",
    ]
    + [r"cust_request_qty_diff_*"],
}

# busco todas las columnas que empiezan con prod_ y agrego key y valor en transformation
for col in numeric_cols:
    if col.startswith("prod_"):
        transformations[col] = [r"{}$".format(col)]
transformations

{'tn': ['tn$',
  'cust_request_qty_per_tn$',
  'tn_lag_*',
  'tn_rolling_mean_*',
  'tn_rolling_max_*',
  'tn_rolling_min_*',
  'tn_.*_vendidas$',
  'tn_agg*',
  'stock_final$',
  'cust_request_tn_minus_tn$',
  'tn_diff_*'],
 'cust_request_qty': ['cust_request_qty$',
  'cust_request_qty_lag_*',
  'cust_request_qty_rolling_mean_*',
  'cust_request_qty_rolling_max_*',
  'cust_request_qty_rolling_min_*',
  'cust_request_qty_.*_vendidas$',
  'cust_request_qty_agg*',
  'cust_request_qty_diff_*'],
 'prod_cust_request_qty_customer_id_vendidas_x_cust_request_qty_cat1_PC': ['prod_cust_request_qty_customer_id_vendidas_x_cust_request_qty_cat1_PC$'],
 'prod_cust_request_qty_customer_id_vendidas_x_cust_request_qty_cat3_vendidas': ['prod_cust_request_qty_customer_id_vendidas_x_cust_request_qty_cat3_vendidas$'],
 'prod_cust_request_qty_customer_id_vendidas_x_cust_request_qty_brand_vendidas': ['prod_cust_request_qty_customer_id_vendidas_x_cust_request_qty_brand_vendidas$'],
 'prod_cust_request_qty_cus

In [51]:
# hago el scaling que sera column/std(col_ref) para cada columna

# primero calculo el std por product_id y customer_id por cada key de transformation
# gruped_std tiene doble indice product_id y customer_id
prod_stats = df.groupby(["product_id", "customer_id"])[
    list(transformations.keys())
].agg(["std"])
prod_stats.columns = [
    f"{col[0]}_{col[1]}" for col in prod_stats.columns
]  # renombro las columnas para que no tengan tupla
prod_stats = prod_stats.reset_index()
prod_stats

# procedo al scaling
for trainer, regex_cols in transformations.items():
    for col in regex_cols:
        # Usar regex para seleccionar las columnas que coinciden
        # chequear si la columna es un regex
        matching_cols = [c for c in numeric_cols if re.match(col, c)]
        if not matching_cols:
            continue  # Si no hay columnas que coincidan, saltar

        # Calcular la media y desviación estándar para cada
        print(f"Processing trainer: {trainer} with columns: {matching_cols}")
        # Escalar las columnas
        for col in matching_cols:
            df = df.merge(
                prod_stats[["product_id", "customer_id", trainer + "_std"]],
                on=["product_id", "customer_id"],
                how="left",
            )
            df[f"{col}_scaled"] = (df[col] / df[trainer + "_std"]).fillna(0)
            # replace nan with 0
            df.drop(columns=[trainer + "_std"], inplace=True)
# elimino las columnas que no son necesarias

Processing trainer: tn with columns: ['tn']
Processing trainer: tn with columns: ['tn_lag_1', 'tn_lag_2', 'tn_lag_3', 'tn_lag_11', 'tn_lag_15']
Processing trainer: tn with columns: ['tn_rolling_mean_6', 'tn_rolling_mean_12']
Processing trainer: tn with columns: ['tn_rolling_max_6', 'tn_rolling_max_12', 'tn_rolling_max_24']
Processing trainer: tn with columns: ['tn_rolling_min_6', 'tn_rolling_min_12', 'tn_rolling_min_24']
Processing trainer: tn with columns: ['tn_cat1_vendidas', 'tn_cat2_vendidas', 'tn_cat3_vendidas', 'tn_brand_vendidas', 'tn_sku_size_vendidas', 'tn_product_id_vendidas', 'tn_customer_id_vendidas', 'tn_customer_vendidas', 'tn_total_vendidas', 'tn_product_vendidas']
Processing trainer: tn with columns: ['stock_final']
Processing trainer: tn with columns: ['tn_diff_1', 'tn_diff_2', 'tn_diff_3', 'tn_diff_5', 'tn_diff_11']
Processing trainer: cust_request_qty with columns: ['cust_request_qty']
Processing trainer: cust_request_qty with columns: ['cust_request_qty_lag_1', 'cus

KeyboardInterrupt: 

In [ ]:
df

,product_id,customer_id,fecha,periodo_min_producto,periodo_max_producto,periodo_min_customer,periodo_max_customer,plan_precios_cuidados,cust_request_qty,cust_request_tn,...,prod_cust_request_qty_cat1_HC_x_cust_request_qty_cat1_FOODS_scaled,prod_cust_request_qty_cat1_HC_x_tn_total_vendidas_scaled,prod_cust_request_qty_cat1_HC_x_tn_customer_id_vendidas_scaled,prod_cust_request_qty_cat1_HC_x_tn_customer_vendidas_scaled,prod_cust_request_qty_cat1_FOODS_x_tn_total_vendidas_scaled,prod_cust_request_qty_cat1_FOODS_x_tn_customer_id_vendidas_scaled,prod_cust_request_qty_cat1_FOODS_x_tn_customer_vendidas_scaled,prod_tn_total_vendidas_x_tn_customer_id_vendidas_scaled,prod_tn_total_vendidas_x_tn_customer_vendidas_scaled,prod_tn_customer_id_vendidas_x_tn_customer_vendidas_scaled
0,20001,0,2017-01,NaT,NaT,NaT,NaT,NaN,286,286.887787,...,2.799457,3.054424,2.710915,2.710915,2.487089,2.228051,2.228051,2.192529,2.192529,1.914635
1,20001,10001,2017-01,2017-01-01,2019-12-01,2017-01-01,2019-12-01,0.0,11,99.438606,...,2.799457,3.054424,2.425737,2.425737,2.487089,2.010139,2.010139,1.920444,1.920444,1.353199
2,20001,10002,2017-01,2017-01-01,2019-12-01,2017-01-01,2019-12-01,0.0,17,38.683010,...,2.799457,3.054424,2.987954,2.987954,2.487089,2.714823,2.714823,2.790461,2.790461,2.528365
3,20001,10003,2017-01,2017-01-01,2019-12-01,2017-01-01,2019-12-01,0.0,17,143.494263,...,2.799457,3.054424,3.646098,3.646098,2.487089,3.007185,3.007185,3.090138,3.090138,3.259741
4,20001,10004,2017-01,2017-01-01,2019-12-01,2017-01-01,2019-12-01,0.0,9,184.729263,...,2.799457,3.054424,2.478936,2.478936,2.487089,2.164907,2.164907,2.002871,2.002871,1.528523
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
346737,21276,10006,2019-12,2019-03-01,2019-12-01,2017-01-01,2019-12-01,0.0,0,0.000000,...,1.561819,2.791784,1.144040,1.144040,1.661655,0.772719,0.772719,1.141480,1.141480,0.455223
346738,21276,10007,2019-12,2019-03-01,2019-12-01,2017-01-01,2019-12-01,0.0,0,0.000000,...,1.561819,2.791784,2.762812,2.762812,1.661655,1.779883,1.779883,2.724013,2.724013,2.034538
346739,21276,10008,2019-12,2019-03-01,2019-12-01,2017-01-01,2019-12-01,0.0,0,0.000000,...,1.561819,2.791784,2.240769,2.240769,1.661655,1.542093,1.542093,2.123499,2.123499,1.360268
346740,21276,10009,2019-12,2019-03-01,2019-12-01,2017-01-01,2019-12-01,0.0,0,0.000000,...,1.561819,2.791784,1.162950,1.162950,1.661655,0.720699,0.720699,1.092442,1.092442,0.403685


In [ ]:
# creo el target
df["target"] = df.groupby(["product_id", "customer_id"])["tn"].shift(-2)
# elimino las rows donde target es nan
df = df[df["target"].notna()]
# hago el scaling del target por tn_std
df = df.merge(
    prod_stats[["product_id", "customer_id", "tn_std"]],
    on=["product_id", "customer_id"],
    how="left",
)
df["target_scaled"] = (df["target"] / df["tn_std"]).fillna(0)
# replace nan with 0
df.drop(columns=["tn_std"], inplace=True)


In [ ]:
# entreno el modelo xgb
# para ello uso ultimo mes como test de optimizacion
# reemplazo en df los -inf y +inf por nan
df.replace([np.inf, -np.inf], np.nan, inplace=True)

last_date_id = df["date_id"].max()
train_df = df[df["date_id"] < last_date_id]
test_df = df[df["date_id"] == last_date_id]


# eliminar columnas que son datatime y transformar objects a categoricos
columns = train_df.select_dtypes(include=["object"]).columns.tolist()
for col in columns:
    if col not in ["product_id", "customer_id", "date_id"]:
        train_df[col] = train_df[col].astype("category")
        test_df[col] = test_df[col].astype("category")
# eliminar columnas que son datatime
datetime_cols = train_df.select_dtypes(include=["datetime"]).columns.tolist()
for col in datetime_cols:
    if col not in ["date_id"]:
        train_df.drop(columns=[col], inplace=True)
        test_df.drop(columns=[col], inplace=True)

# elimino todas las columnas que no digan _std
#columns_to_drop = [
#    col for col in train_df.columns if not col.endswith("_scaled") and col != "target"
#]
columns_to_drop = []

# creo los datasets de entrenamiento y test
dtrain = xgb.DMatrix(
    train_df.drop(columns=set(columns_to_drop+["fecha", "target_scaled", "target", "date_id"])),
    label=train_df["target_scaled"],
    enable_categorical=True,
)
dtest = xgb.DMatrix(
    test_df.drop(columns=set(columns_to_drop + ["fecha", "target_scaled", "target", "date_id"])),
    label=test_df["target_scaled"],
    enable_categorical=True,
)
# entreno el modelo
params = {
    "objective": "reg:tweedie",
    "tweedie_variance_power": 1.2,
    "verbosity": 1,
    #"booster": "gblinear"
}
model = xgb.train(
    params,
    dtrain,
    num_boost_round=1090,
    evals=[(dtrain, "train"), (dtest, "test")],
    early_stopping_rounds=1000,
    verbose_eval=True,
)

/tmp/ipykernel_28035/2188573832.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df[col] = train_df[col].astype("category")
/tmp/ipykernel_28035/2188573832.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df[col] = test_df[col].astype("category")
/tmp/ipykernel_28035/2188573832.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydat

[0]	train-tweedie-nloglik@1:nan	test-tweedie-nloglik@1:nan
[1]	train-tweedie-nloglik@1:nan	test-tweedie-nloglik@1:nan
[2]	train-tweedie-nloglik@1:nan	test-tweedie-nloglik@1:nan
[3]	train-tweedie-nloglik@1:nan	test-tweedie-nloglik@1:nan
[4]	train-tweedie-nloglik@1:nan	test-tweedie-nloglik@1:nan
[5]	train-tweedie-nloglik@1:nan	test-tweedie-nloglik@1:nan
[6]	train-tweedie-nloglik@1:nan	test-tweedie-nloglik@1:nan
[7]	train-tweedie-nloglik@1:nan	test-tweedie-nloglik@1:nan
[8]	train-tweedie-nloglik@1:nan	test-tweedie-nloglik@1:nan
[9]	train-tweedie-nloglik@1:nan	test-tweedie-nloglik@1:nan
[10]	train-tweedie-nloglik@1:nan	test-tweedie-nloglik@1:nan
[11]	train-tweedie-nloglik@1:nan	test-tweedie-nloglik@1:nan
[12]	train-tweedie-nloglik@1:nan	test-tweedie-nloglik@1:nan
[13]	train-tweedie-nloglik@1:nan	test-tweedie-nloglik@1:nan
[14]	train-tweedie-nloglik@1:nan	test-tweedie-nloglik@1:nan
[15]	train-tweedie-nloglik@1:nan	test-tweedie-nloglik@1:nan
[16]	train-tweedie-nloglik@1:nan	test-tweedie-nlog

KeyboardInterrupt: 

In [41]:
predictions = model.predict(dtest)
test_df["predictions"] = predictions
# hago la transformada inversa de las predicciones
test_df = test_df.merge(
    prod_stats[["product_id", "customer_id", "tn_std"]],
    on=["product_id", "customer_id"],
    how="left",
)
test_df["predictions"] = test_df["predictions"] * test_df["tn_std"]
test_df

/tmp/ipykernel_28035/819748722.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df["predictions"] = predictions


,product_id,customer_id,fecha,plan_precios_cuidados,cust_request_qty,cust_request_tn,tn,stock_final,cat1,cat2,...,prod_cust_request_qty_cat1_FOODS_x_tn_total_vendidas_scaled,prod_cust_request_qty_cat1_FOODS_x_tn_customer_id_vendidas_scaled,prod_cust_request_qty_cat1_FOODS_x_tn_customer_vendidas_scaled,prod_tn_total_vendidas_x_tn_customer_id_vendidas_scaled,prod_tn_total_vendidas_x_tn_customer_vendidas_scaled,prod_tn_customer_id_vendidas_x_tn_customer_vendidas_scaled,target,target_scaled,predictions,tn_std
0,20001,0,2019-10,NaN,198,630.006104,614.043091,185.27153,HC,ROPA LAVADO,...,2.404575,2.252553,2.252553,2.709321,2.709321,2.474023,789.871338,4.694554,554.176208,168.252670
1,20001,10001,2019-10,0.0,21,178.494263,176.029800,185.27153,HC,ROPA LAVADO,...,2.404575,2.151625,2.151625,2.512508,2.512508,1.960022,180.219376,1.699657,152.428146,106.032784
2,20001,10002,2019-10,0.0,10,19.435640,17.408060,185.27153,HC,ROPA LAVADO,...,2.404575,1.801407,1.801407,2.263140,2.263140,1.407338,113.331650,3.421672,27.877583,33.121716
3,20001,10003,2019-10,0.0,6,76.006248,76.006248,185.27153,HC,ROPA LAVADO,...,2.404575,2.128980,2.128980,2.673958,2.673958,2.065496,102.275169,1.473470,108.659027,69.411079
4,20001,10004,2019-10,0.0,12,327.919067,324.961731,185.27153,HC,ROPA LAVADO,...,2.404575,2.574656,2.574656,2.911376,2.911376,2.733068,34.648102,0.303205,166.660019,114.272812
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10148,21276,10006,2019-10,0.0,0,0.000000,0.000000,1.05889,PC,PIEL1,...,3.800269,3.170675,3.170675,4.084041,4.084041,2.922154,0.000000,0.000000,0.000000,0.000000
10149,21276,10007,2019-10,0.0,0,0.000000,0.000000,1.05889,PC,PIEL1,...,3.800269,4.428898,4.428898,5.910235,5.910235,4.802774,0.000000,0.000000,0.000000,0.000000
10150,21276,10008,2019-10,0.0,0,0.000000,0.000000,1.05889,PC,PIEL1,...,3.800269,4.222029,4.222029,5.069371,5.069371,3.887443,0.000000,0.000000,0.000000,0.000000
10151,21276,10009,2019-10,0.0,0,0.000000,0.000000,1.05889,PC,PIEL1,...,3.800269,3.459329,3.459329,4.572227,4.572227,3.545981,0.000000,0.000000,0.000000,0.000000


In [44]:
test_df_grouped = test_df.groupby(["product_id"]).agg(
    {
        "predictions": "sum",
        "target": "sum",
    }
).reset_index()
product_ids = pd.read_csv(DATA_FOLDER + "product_id_apredecir201912.txt", sep="\t")[
    "product_id"
].tolist()
test_df_grouped = test_df_grouped[test_df_grouped["product_id"].isin(product_ids)]

total_error = np.sum(np.abs(test_df_grouped["predictions"] - test_df_grouped["target"])) / np.sum(test_df_grouped["target"])
print(f"Total error: {total_error:.4f}")
test_df_grouped

Total error: 0.2826


,product_id,predictions,target
0,20001,1309.355103,1504.688599
1,20002,982.764648,1087.308594
2,20003,895.966553,892.501282
3,20004,712.544434,637.900024
4,20005,722.896667,593.244446
...,...,...,...
916,21263,0.059175,0.012700
918,21265,0.072411,0.050070
919,21266,0.069342,0.051210
920,21267,0.060921,0.015690
